In [19]:
# ==============================================================================
# 1. IMPORTS
# ==============================================================================

# OpenPIV
from openpiv import windef
from openpiv import tools, scaling, validation, filters, preprocess
import openpiv.pyprocess as process
from openpiv import pyprocess

# General
import numpy as np
import tifffile as tif
import napari
import matplotlib.pyplot as plt
import pandas as pd
from roifile import ImagejRoi

import pathlib
from time import time
import warnings
%matplotlib qt

# Custom functions
from src.PIV import run_PIV_on_frames

In [6]:
# ==============================================================================
# Paths
# ==============================================================================

unstressed_reference__path = '/mnt/crunch/Clark/Fly_TFM/data/second/unstressed_reference/'
sitting_reference_path = '/mnt/crunch/Clark/Fly_TFM/data/second/sitting_reference/'
time_resolved_path = '/mnt/crunch/Clark/Fly_TFM/data/second/time_resolved/'

In [20]:
# ==============================================================================
# Do subtraction using sitting reference
# ==============================================================================

sitting_reference = pd.read_csv('/mnt/crunch/Clark/Fly_TFM/data/second/sitting_reference/PIVlab_0001.txt', skiprows=2)

starting_frame = 1
num_frames = 28
num_gridpoints = len(sitting_reference)

u_subtraction = np.zeros((num_frames, num_gridpoints))
v_subtraction = np.zeros((num_frames, num_gridpoints))
    
for index, current_frame in enumerate(range(starting_frame, num_frames + 1)):
        
    # Load data
    data = pd.read_csv(sitting_reference_path + f'PIVlab_{current_frame:04d}.txt', skiprows=2)
        
    # Load columns and inject columns into subtaction method
    u_subtraction[index] = data['u [px/frame]'].values - sitting_reference['u [px/frame]'].values
    v_subtraction[index] = data['v [px/frame]'].values - sitting_reference['v [px/frame]'].values

In [21]:
# ==============================================================================
# Integrate starting from sitting reference
# ==============================================================================

#NOTE Remember, need to add a frame manually before every other frame for visualization of integrated displacements

u_time_resolved = np.zeros((num_frames - 1, num_gridpoints))
v_time_resolved = np.zeros((num_frames - 1, num_gridpoints))

# Load .csv columns into numpy arrays
for index, current_frame in enumerate(range(1, num_frames)):
    
    # Load data
    data = pd.read_csv(time_resolved_path + f'PIVlab_{current_frame:04d}.txt', skiprows=2)
    
    # Load columns of currently integrated displacement
    u_time_resolved[index] = data['u [px/frame]'].values
    v_time_resolved[index] = data['v [px/frame]'].values
    
# Integrate
u_integrated = np.cumsum(u_time_resolved, axis=0)
v_integrated = np.cumsum(v_time_resolved, axis=0)

In [ ]:
# ==============================================================================
# Visualize the subtraction method
# ==============================================================================

u_grid = u_subtraction.reshape(28, 63, 63)
v_grid = v_subtraction.reshape(28, 63, 63)
z_grid = np.zeros_like(u_grid) # napari expects num spatial dimensions = num vector components, so manually set dz = 0

vector_grid = np.stack([z_grid, v_grid, u_grid], axis=-1)
magnitude_grid = np.sqrt(u_grid**2 + v_grid**2)

# ------------------------------------------------------------------------------

viewer = napari.Viewer()

magnitude_layer = viewer.add_image(
    magnitude_grid,
    name='Magnitude Heatmap',
    colormap='jet',
    #interpolation2d='bicubic',
    contrast_limits=[0, 3]
)

vector_layer = viewer.add_vectors(
    vector_grid,
    name="Displacements",
    vector_style='arrow',
    edge_width=0.2,
    length=1.0,
    edge_color="black"
)

magnitude_layer.colorbar.visible = True

napari.run()

In [36]:
# ==============================================================================
# Visualize the integration method
# ==============================================================================

u_grid = u_integrated.reshape(27, 63, 63)
v_grid = v_integrated.reshape(27, 63, 63)
z_grid = np.zeros_like(u_grid)

vector_grid = np.stack([z_grid, v_grid, u_grid], axis=-1)
magnitude_grid = np.sqrt(u_grid**2 + v_grid**2)

# ------------------------------------------------------------------------------

viewer = napari.Viewer()

magnitude_layer = viewer.add_image(
    magnitude_grid,
    name='Magnitude Heatmap',
    colormap='jet',
    #interpolation2d='bicubic',
    contrast_limits=[0, 3]
)

vector_layer = viewer.add_vectors(
    vector_grid,
    name="Displacements",
    vector_style='arrow',
    edge_width=0.2,
    length=1.0,
    edge_color="black"
)

magnitude_layer.colorbar.visible = True

napari.run()

In [ ]:
# ==============================================================================
# 1. Prepare Integration Data (27 frames -> shifted to 28 frames)
# ==============================================================================
u_grid_integrated_raw = u_integrated.reshape(27, 63, 63)
v_grid_integrated_raw = v_integrated.reshape(27, 63, 63)
z_grid_integrated_raw = np.zeros_like(u_grid_integrated_raw)

# Prepend 1 frame of zeros to the front (axis=0) so frame 0 is empty
u_grid_integrated = np.pad(u_grid_integrated_raw, ((1, 0), (0, 0), (0, 0)), mode='constant', constant_values=0)
v_grid_integrated = np.pad(v_grid_integrated_raw, ((1, 0), (0, 0), (0, 0)), mode='constant', constant_values=0)
z_grid_integrated = np.pad(z_grid_integrated_raw, ((1, 0), (0, 0), (0, 0)), mode='constant', constant_values=0)

vector_int = np.stack([z_grid_integrated, v_grid_integrated, u_grid_integrated], axis=-1)
magnitude_int = np.sqrt(u_grid_integrated**2 + v_grid_integrated**2)

# ==============================================================================
# 2. Prepare Subtraction Data (28 frames)
# ==============================================================================
u_sub = u_subtraction.reshape(28, 63, 63)
v_sub = v_subtraction.reshape(28, 63, 63)
z_sub = np.zeros_like(u_sub)

vector_sub = np.stack([z_sub, v_sub, u_sub], axis=-1)
magnitude_sub = np.sqrt(u_sub**2 + v_sub**2)

# ==============================================================================
# 3. Launch single viewer with both datasets
# ==============================================================================
viewer = napari.Viewer()

# --- Integration Method Layers ---
mag_int_layer = viewer.add_image(
    magnitude_int,
    name='Integration - Heatmap',
    colormap='jet',
    contrast_limits=[0, 3]
)

vec_int_layer = viewer.add_vectors(
    vector_int,
    name='Integration - Vectors',
    vector_style='arrow',
    edge_width=0.2,
    length=1.0,
    edge_color='black'
)

# --- Subtraction Method Layers ---
mag_sub_layer = viewer.add_image(
    magnitude_sub,
    name='Subtraction - Heatmap',
    colormap='jet',
    contrast_limits=[0, 3]
)

vec_sub_layer = viewer.add_vectors(
    vector_sub,
    name='Subtraction - Vectors',
    vector_style='arrow',
    edge_width=0.2,
    length=1.0,
    edge_color='black'
)

# Colorbar display
mag_sub_layer.colorbar.visible = True

# Side-by-side grid view (2 layers per box)
viewer.grid.enabled = True
viewer.grid.stride = 2

napari.run()